# AM5061 · Week 11 · Transcritical CO₂ booster

**Design of Thermal and Fluid Systems** · Applied Mechanics, IIT Madras · Jul–Nov 2026

Run the two setup cells below once, then work down the notebook. Nothing needs to be installed on your own machine.


## Setup

Run these two cells first. The second one writes the course helper module, so this notebook is self-contained.


In [ ]:
#@title Install the property library  { display-mode: "form" }
!pip install -q CoolProp openpyxl
print('CoolProp ready')


In [ ]:
%%writefile am5061.py
"""AM5061 - Design of Thermal and Fluid Systems.
Shared helpers for the course notebooks.

Design of Thermal and Fluid Systems, IIT Madras, Jul-Nov 2026.

This module is deliberately thin. It wraps CoolProp with names and units that
match the lecture notation, adds an Excel writer that produces workbooks you
can actually filter, and sets a consistent plot style. It does NOT hide the
engineering: every case notebook still writes its own equations.

Install (first cell of any Colab notebook):
    !pip install -q CoolProp openpyxl

Units are SI throughout, with ONE exception that is flagged everywhere it
appears: temperatures in function arguments named `..._C` are in Celsius,
because that is how the case briefs state them. Everything internal is kelvin.
"""
from __future__ import annotations

import math

from CoolProp.CoolProp import PropsSI, PhaseSI

__all__ = [
    "K", "C", "State", "state", "sat_liquid", "sat_vapour", "p_sat", "T_sat",
    "glide", "h_fg", "critical", "fluids", "solve", "sweep", "to_excel",
    "style_plots", "NAVY", "ORANGE", "BLUE", "MUTED",
    "lmtd", "effectiveness", "ntu_required", "exergy", "T0_REF", "P0_REF",
]

# ---------------------------------------------------------------- constants
NAVY, ORANGE, BLUE, MUTED = "#1F3864", "#ED7D31", "#4472C4", "#59626E"
T0 = 273.15


def K(t_celsius: float) -> float:
    """Celsius -> kelvin. Use this at the boundary, never inside a formula."""
    return t_celsius + T0


def C(t_kelvin: float) -> float:
    """Kelvin -> Celsius, for reporting only."""
    return t_kelvin - T0


# ------------------------------------------------------------------- states
class State:
    """A thermodynamic state. Immutable, and it knows its own fluid.

    Construct it with any two independent properties:
        State("R134a", P=1e6, T=K(70))
        State("Water", P=101325, Q=0)      # saturated liquid
        State("R134a", P=p_cond, H=h2)

    Then read properties as attributes: .T .p .h .s .d .cp .x
    Attribute names match the lecture notation, not CoolProp's letter codes,
    so a student reading the notebook does not need the CoolProp manual open.
    """

    _MAP = {"T": "T", "P": "P", "H": "H", "S": "S", "D": "D", "Q": "Q"}

    def __init__(self, fluid: str, **kw):
        if len(kw) != 2:
            raise ValueError(
                f"a state needs exactly two properties, got {list(kw)}. "
                "Two and only two - that is the phase rule, not a quirk."
            )
        (n1, v1), (n2, v2) = kw.items()
        for n in (n1, n2):
            if n not in self._MAP:
                raise ValueError(f"unknown property {n!r}; use T, P, H, S, D or Q")
        self.fluid, self._args = fluid, (n1, v1, n2, v2)

    def _get(self, what: str) -> float:
        n1, v1, n2, v2 = self._args
        return PropsSI(what, n1, v1, n2, v2, self.fluid)

    # Named so they read like the equations on the slides.
    T  = property(lambda s: s._get("T"),  doc="temperature, K")
    p  = property(lambda s: s._get("P"),  doc="pressure, Pa")
    h  = property(lambda s: s._get("H"),  doc="specific enthalpy, J/kg")
    s  = property(lambda s: s._get("S"),  doc="specific entropy, J/kg.K")
    d  = property(lambda s: s._get("D"),  doc="density, kg/m3")
    cp = property(lambda s: s._get("C"),  doc="cp, J/kg.K")
    mu = property(lambda s: s._get("V"),  doc="dynamic viscosity, Pa.s")
    k  = property(lambda s: s._get("L"),  doc="thermal conductivity, W/m.K")
    x  = property(lambda s: s._get("Q"),  doc="vapour quality, - (=-1 if single phase)")

    @property
    def T_C(self) -> float:
        return C(self.T)

    @property
    def phase(self) -> str:
        n1, v1, n2, v2 = self._args
        return PhaseSI(n1, v1, n2, v2, self.fluid)

    def __repr__(self):
        try:
            return (f"State({self.fluid}: {self.T_C:.2f} C, {self.p/1e5:.3f} bar, "
                    f"h={self.h/1e3:.2f} kJ/kg, {self.phase})")
        except Exception:
            return f"State({self.fluid}, {self._args})"


def state(fluid: str, **kw) -> State:
    """Shorthand for State(...)."""
    return State(fluid, **kw)


def sat_liquid(fluid: str, *, T=None, p=None) -> State:
    """Saturated liquid at T or p. Give one, not both."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=0) if T is not None else State(fluid, P=p, Q=0)


def sat_vapour(fluid: str, *, T=None, p=None) -> State:
    """Saturated vapour at T or p."""
    if (T is None) == (p is None):
        raise ValueError("give exactly one of T or p")
    return State(fluid, T=T, Q=1) if T is not None else State(fluid, P=p, Q=1)


def p_sat(fluid: str, T: float) -> float:
    """Saturation pressure, Pa. For a BLEND this is the bubble-point pressure."""
    return PropsSI("P", "T", T, "Q", 0, fluid)


def T_sat(fluid: str, p: float) -> float:
    """Saturation temperature, K.

    WARNING for blends: a zeotropic mixture has no single saturation
    temperature. This returns the BUBBLE point. Use glide() to see the spread.
    """
    return PropsSI("T", "P", p, "Q", 0, fluid)


def glide(fluid: str, p: float) -> float:
    """Dew minus bubble temperature at p, K. Zero for a pure fluid."""
    return (PropsSI("T", "P", p, "Q", 1, fluid)
            - PropsSI("T", "P", p, "Q", 0, fluid))


def h_fg(fluid: str, *, T=None, p=None) -> float:
    """Latent heat, J/kg."""
    return sat_vapour(fluid, T=T, p=p).h - sat_liquid(fluid, T=T, p=p).h


def critical(fluid: str) -> dict:
    """Critical point, for checking you are not extrapolating past it."""
    return {"T": PropsSI("TCRIT", fluid), "p": PropsSI("PCRIT", fluid)}


def fluids() -> list:
    """Every fluid CoolProp knows. There are about 130."""
    import CoolProp
    return sorted(CoolProp.__fluids__)


# ------------------------------------------------------------------ solvers
def solve(f, x0, *, tol=1e-10, max_iter=200, bracket=None):
    """Find x where f(x) = 0.

    Uses Brent's method when you give a bracket (robust, always converges if
    the bracket is valid), otherwise secant from x0. Raises with a readable
    message rather than returning a wrong answer silently, which is the whole
    problem with doing this in a spreadsheet.
    """
    from scipy.optimize import brentq, newton
    if bracket is not None:
        a, b = bracket
        fa, fb = f(a), f(b)
        if fa * fb > 0:
            raise ValueError(
                f"f({a:g})={fa:g} and f({b:g})={fb:g} have the same sign, so no "
                "root is bracketed. Widen the bracket or check the equation."
            )
        return brentq(f, a, b, xtol=tol, maxiter=max_iter)
    return newton(f, x0, tol=tol, maxiter=max_iter)


def sweep(fn, values, *, name="x"):
    """Run fn(v) for each v and collect the results as a list of dicts.

    fn must return a dict. The sweep variable is added under `name`, so the
    result drops straight into to_excel().
    """
    rows = []
    for v in values:
        out = fn(v)
        if not isinstance(out, dict):
            raise TypeError("the swept function must return a dict of results")
        rows.append({name: v, **out})
    return rows


# -------------------------------------------------------------------- excel
def to_excel(path, sheets: dict, *, sources=None, summary=None, title=None):
    """Write a workbook that is genuinely usable.

    Every numeric cell is written as a number (not text), AutoFilter is on,
    the header row is frozen, and columns are sized to content. A Summary and
    a Sources sheet are always present, because a result you cannot trace is
    not an engineering deliverable.

        sheets  = {"Sweep": [ {...}, {...} ], ...}   list of dicts per sheet
        sources = [ ("what", "where it came from"), ... ]
        summary = [ ("quantity", value, "units"), ... ]
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    wb = Workbook()
    wb.remove(wb.active)
    head_font = Font(bold=True, color="FFFFFF", name="Calibri")
    head_fill = PatternFill("solid", fgColor="1F3864")

    def _write(ws, rows, headers=None):
        headers = headers or (list(rows[0].keys()) if rows else [])
        for j, hname in enumerate(headers, 1):
            c = ws.cell(row=1, column=j, value=hname)
            c.font, c.fill = head_font, head_fill
            c.alignment = Alignment(horizontal="left")
        for i, row in enumerate(rows, 2):
            for j, hname in enumerate(headers, 1):
                v = row.get(hname)
                # Numbers stay numbers. This is the single most common way a
                # delivered workbook turns out not to be filterable.
                if isinstance(v, bool):
                    v = str(v)
                elif isinstance(v, (int, float)) and not isinstance(v, bool):
                    v = float(v) if isinstance(v, float) else v
                ws.cell(row=i, column=j, value=v)
        if rows:
            ws.auto_filter.ref = (f"A1:{get_column_letter(len(headers))}"
                                  f"{len(rows) + 1}")
        ws.freeze_panes = "A2"
        for j, hname in enumerate(headers, 1):
            width = max([len(str(hname))] +
                        [len(f"{r.get(hname)}") for r in rows[:200]]) + 3
            ws.column_dimensions[get_column_letter(j)].width = min(width, 42)

    # Summary first, so it is what opens.
    ws = wb.create_sheet("Summary")
    ws["A1"] = title or "AM5061 results"
    ws["A1"].font = Font(bold=True, size=14, color="1F3864")
    r = 3
    for item in (summary or []):
        for j, v in enumerate(item, 1):
            ws.cell(row=r, column=j, value=v)
        r += 1
    ws.column_dimensions["A"].width = 46
    ws.column_dimensions["B"].width = 18
    ws.column_dimensions["C"].width = 14

    for sname, rows in sheets.items():
        _write(wb.create_sheet(sname[:31]), rows)

    ws = wb.create_sheet("Sources")
    _write(ws, [{"item": a, "source": b} for a, b in (sources or [])])

    wb.save(path)
    return path


# --------------------------------------------------------------- plot style
def style_plots():
    """Match the lecture decks, so figures in a report look like the slides."""
    import matplotlib as mpl
    mpl.rcParams.update({
        "figure.figsize": (7.2, 4.4), "figure.dpi": 110,
        "axes.edgecolor": MUTED, "axes.labelcolor": NAVY,
        "axes.titlecolor": NAVY, "axes.titlesize": 11.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "font.size": 10, "legend.frameon": False,
        "axes.prop_cycle": mpl.cycler(color=[NAVY, ORANGE, BLUE, "#7F9DB9"]),
    })


# ------------------------------------------------- exchanger relations
def lmtd(dT1, dT2):
    """Log-mean temperature difference.

    Falls back to the arithmetic mean when the two ends are within 1% of each
    other, where the log form is numerically unstable and the two agree to
    better than 0.01% anyway.
    """
    dT1, dT2 = float(dT1), float(dT2)
    if dT1 <= 0 or dT2 <= 0:
        raise ValueError(
            f"temperature difference must be positive at both ends "
            f"(got {dT1:g} and {dT2:g}). A non-positive end means the streams "
            "cross, which no exchanger of this configuration can do."
        )
    if abs(dT1 - dT2) < 0.01*max(dT1, dT2):
        return 0.5*(dT1 + dT2)
    return (dT1 - dT2)/math.log(dT1/dT2)


def effectiveness(config, NTU, Cr):
    """Effectiveness for the standard configurations.

    config: 'counter', 'parallel', 'shell1'  (one shell pass, 2/4/... tube passes),
            'cross-both-unmixed' (approximate), 'cross-Cmax-mixed', 'cross-Cmin-mixed'

    Cr = C_min/C_max. Cr = 0 is the phase-change limit and every configuration
    collapses to the same expression, which is why boilers and condensers are
    easy and everything else is not.
    """
    if NTU < 0:
        raise ValueError("NTU cannot be negative")
    if not 0 <= Cr <= 1:
        raise ValueError(f"Cr must be between 0 and 1, got {Cr:g}")
    if Cr == 0:                       # phase change on one side
        return 1 - math.exp(-NTU)
    if config == "counter":
        if abs(Cr - 1) < 1e-12:
            return NTU/(1 + NTU)
        e = math.exp(-NTU*(1 - Cr))
        return (1 - e)/(1 - Cr*e)
    if config == "parallel":
        return (1 - math.exp(-NTU*(1 + Cr)))/(1 + Cr)
    if config == "shell1":
        r = math.sqrt(1 + Cr*Cr)
        e = math.exp(-NTU*r)
        return 2/(1 + Cr + r*(1 + e)/(1 - e))
    if config == "cross-both-unmixed":
        return 1 - math.exp((math.exp(-Cr*NTU**0.78) - 1)*NTU**0.22/Cr)
    if config == "cross-Cmax-mixed":
        return (1/Cr)*(1 - math.exp(-Cr*(1 - math.exp(-NTU))))
    if config == "cross-Cmin-mixed":
        return 1 - math.exp(-(1 - math.exp(-Cr*NTU))/Cr)
    raise ValueError(f"unknown configuration {config!r}")


def ntu_required(config, eps, Cr, hi=200.0):
    """Invert effectiveness() for NTU. Design direction, rather than rating."""
    eps_max = effectiveness(config, hi, Cr)
    if eps >= eps_max:
        raise ValueError(
            f"effectiveness {eps:g} is unreachable for {config} at Cr={Cr:g}; "
            f"the limit as NTU->infinity is {eps_max:.6f}. "
            "Change the configuration or accept less."
        )
    return solve(lambda n: effectiveness(config, n, Cr) - eps, 1.0,
                 bracket=(1e-9, hi))


# --------------------------------------------------------------- exergy
T0_REF, P0_REF = 303.15, 101325.0      # 30 C, sea level: the Chennai dead state


def exergy(st, T0=T0_REF, p0=P0_REF):
    """Specific flow exergy, J/kg:  (h - h0) - T0*(s - s0).

    The dead state is the ambient the plant actually sits in, so it is a
    DESIGN CHOICE, not a constant. Report which one you used - a Chennai
    dead state and a European one give different answers for the same plant.
    """
    ref = State(st.fluid, T=T0, P=p0)
    return (st.h - ref.h) - T0*(st.s - ref.s)


---
## The case

A **transcritical CO₂ booster** for a supermarket cold chain in **Chennai**.
Two evaporating levels: **−30 °C frozen** and **−8 °C chilled**. CO₂ throughout.

Deliverable **D-11**: the **optimum gas-cooler pressure** as a function of
ambient temperature over 25–42 °C. Plot COP against P_gc and mark the optimum
locus.

### Why CO₂ has no condenser

CO₂'s critical point is **31.0 °C at 73.8 bar**. In Chennai the ambient is above
that for much of the year, so on the high side the CO₂ **never condenses**. It
is cooled, as a single dense phase, in a **gas cooler**.

That changes the design completely. With no condensation there is no saturation
line tying pressure to temperature, so **the high-side pressure becomes a free
variable** — and there is an optimum. Finding it is this week's work.

> This case was **impossible** on the previous toolchain: MSL carries CO₂ only
> as an ideal gas, which cannot represent transcritical behaviour at all.


## 1. Why an optimum exists

In [ ]:
import am5061 as am
import numpy as np, matplotlib.pyplot as plt
from CoolProp.CoolProp import PropsSI
from scipy.optimize import brentq, minimize_scalar
am.style_plots()

F = "CO2"
crit = am.critical(F)
print(f"  CO2 critical point: {am.C(crit['T']):.2f} C, {crit['p']/1e5:.2f} bar")
print(f"  Chennai design ambient 35 C is ABOVE that, so the high side is")
print(f"  transcritical and there is no condenser.\n")

T_gc_out = am.K(38)          # gas cooler exit, 35 C ambient + 3 K approach
print(f"{'P_gc, bar':>10}{'h at exit, kJ/kg':>19}{'rho, kg/m3':>13}")
for P in (75, 80, 85, 90, 95, 100, 110, 120):
    h = PropsSI("H","P",P*1e5,"T",T_gc_out,F)/1e3
    d = PropsSI("D","P",P*1e5,"T",T_gc_out,F)
    print(f"{P:10.0f}{h:19.2f}{d:13.2f}")
print("\n  Raising the pressure costs compressor work, but it also drops the")
print("  gas-cooler exit enthalpy STEEPLY near the pseudo-critical region,")
print("  which buys refrigeration effect. Those two fight, so there is a peak.")


## 2. The booster architecture

```
   LT evaporator (-30 C) -> LT compressor -+
                                            |--> MT suction -> MT compressor -> gas cooler
   MT evaporator (-8 C)  ------------------+                                        |
                                            flash gas from receiver <--- expansion <-+
```

The LT compressor discharges into the MT suction rather than to the gas cooler.
The receiver flash gas joins there too. One high-stage machine handles the lot.


In [ ]:
T_LT, T_MT = am.K(-30), am.K(-8)
Q_LT, Q_MT = 30e3, 90e3          # W, frozen and chilled cabinet loads
p_rec      = 38e5                # receiver pressure, typical
eta_LT, eta_MT = 0.65, 0.70      # isentropic efficiencies
dT_sup     = 5.0                 # useful superheat at both evaporators

p_LT, p_MT = am.p_sat(F, T_LT), am.p_sat(F, T_MT)
print(f"  LT evaporating  {am.C(T_LT):6.1f} C -> {p_LT/1e5:6.2f} bar")
print(f"  MT evaporating  {am.C(T_MT):6.1f} C -> {p_MT/1e5:6.2f} bar")
print(f"  receiver                        {p_rec/1e5:6.2f} bar")

def compress(p_in, h_in, p_out, eta):
    s_in = PropsSI("S","P",p_in,"H",h_in,F)
    h_s  = PropsSI("H","P",p_out,"S",s_in,F)
    return h_in + (h_s - h_in)/eta

def booster(P_gc, T_amb=am.K(35), approach=3.0):
    """Steady-state booster cycle. Returns duties, work and COP."""
    T_gc = T_amb + approach
    h_gc = PropsSI("H","P",P_gc,"T",T_gc,F)          # gas cooler exit

    # expansion to the receiver: flash separates liquid and vapour
    h_rec  = h_gc                                     # isenthalpic
    h_f    = PropsSI("H","P",p_rec,"Q",0,F)
    h_g    = PropsSI("H","P",p_rec,"Q",1,F)
    x_flash = min(max((h_rec - h_f)/(h_g - h_f), 0.0), 1.0)

    # LT circuit
    h_LT_in  = h_f                                    # liquid from receiver
    h_LT_out = PropsSI("H","P",p_LT,"T",T_LT+dT_sup,F)
    m_LT     = Q_LT/(h_LT_out - h_LT_in)
    h_LT_dis = compress(p_LT, h_LT_out, p_MT, eta_LT)
    W_LT     = m_LT*(h_LT_dis - h_LT_out)

    # MT circuit
    h_MT_in  = h_f
    h_MT_out = PropsSI("H","P",p_MT,"T",T_MT+dT_sup,F)
    m_MT     = Q_MT/(h_MT_out - h_MT_in)

    # MT suction mixes: MT evaporator + LT discharge + receiver flash gas
    m_liq   = m_LT + m_MT
    m_total = m_liq/max(1 - x_flash, 1e-6)            # total through the gas cooler
    m_flash = m_total - m_liq
    h_mix   = (m_MT*h_MT_out + m_LT*h_LT_dis + m_flash*h_g)/m_total
    h_MT_dis = compress(p_MT, h_mix, P_gc, eta_MT)
    W_MT     = m_total*(h_MT_dis - h_mix)

    return {"P_gc, bar": P_gc/1e5, "T_amb, C": am.C(T_amb), "T_gc_out, C": am.C(T_gc),
            "h_gc, kJ/kg": h_gc/1e3, "flash fraction": x_flash,
            "m_LT, kg/s": m_LT, "m_MT, kg/s": m_MT, "m_total, kg/s": m_total,
            "W_LT, kW": W_LT/1e3, "W_MT, kW": W_MT/1e3,
            "W_total, kW": (W_LT+W_MT)/1e3,
            "COP": (Q_LT+Q_MT)/(W_LT+W_MT),
            "discharge, C": am.C(PropsSI("T","P",P_gc,"H",h_MT_dis,F))}

r = booster(90e5)
for k, v in r.items(): print(f"  {k:16s} {v:10.4f}")


## 3. The optimum, at design ambient

In [ ]:
Ps = np.arange(75e5, 131e5, 0.5e5)
cop = [booster(P)["COP"] for P in Ps]
i = int(np.argmax(cop))
# A three-point bracket is fragile here because the peak moves with ambient.
# A bounded search over the physically sensible range always works.
res = minimize_scalar(lambda P: -booster(P)["COP"],
                      bounds=(75e5, 130e5), method="bounded")
P_opt = res.x

print(f"  grid maximum      P_gc = {Ps[i]/1e5:.1f} bar, COP = {cop[i]:.4f}")
print(f"  refined optimum   P_gc = {P_opt/1e5:.2f} bar, COP = {booster(P_opt)['COP']:.4f}")
print(f"\n  COP at 80 bar  {booster(80e5)['COP']:.4f}")
print(f"  COP at optimum {booster(P_opt)['COP']:.4f}")
print(f"  COP at 120 bar {booster(120e5)['COP']:.4f}")
print("\n  Running 10 bar off the optimum costs real energy. On a supermarket")
print("  that runs 8760 h a year, this single set-point is worth optimising.")


In [ ]:
fig, ax = plt.subplots()
for T_amb_C in (25, 30, 35, 40, 42):
    cs = [booster(P, T_amb=am.K(T_amb_C))["COP"] for P in Ps]
    ax.plot(Ps/1e5, cs, lw=2.2, label=f"{T_amb_C} °C ambient")
    j = int(np.argmax(cs))
    ax.plot(Ps[j]/1e5, cs[j], "o", ms=8, color=am.NAVY, zorder=5)
ax.set_xlabel("gas cooler pressure  (bar)"); ax.set_ylabel("system COP")
ax.set_title("Each ambient has its own optimum. The peaks trace a locus.")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 4. The optimum locus — the deliverable

In [ ]:
rows = []
for T_amb_C in np.arange(25, 43, 1.0):
    res = minimize_scalar(lambda P: -booster(P, T_amb=am.K(T_amb_C))["COP"],
                          bounds=(75e5, 130e5), method="bounded")
    b = booster(res.x, T_amb=am.K(T_amb_C))
    rows.append({"T_ambient, C": float(T_amb_C), "P_gc optimum, bar": b["P_gc, bar"],
                 "COP at optimum": b["COP"], "W_total, kW": b["W_total, kW"],
                 "discharge, C": b["discharge, C"],
                 "flash fraction": b["flash fraction"]})

print(f"{'T_amb C':>9}{'P_opt bar':>11}{'COP':>9}{'W kW':>9}{'discharge C':>13}")
for r_ in rows[::3]:
    print(f"{r_['T_ambient, C']:9.0f}{r_['P_gc optimum, bar']:11.2f}"
          f"{r_['COP at optimum']:9.4f}{r_['W_total, kW']:9.2f}{r_['discharge, C']:13.1f}")

# a linear rule of thumb, fitted to the locus
T = np.array([r_["T_ambient, C"] for r_ in rows])
P = np.array([r_["P_gc optimum, bar"] for r_ in rows])
a, b_ = np.polyfit(T, P, 1)
print(f"\n  fitted rule of thumb:  P_gc,opt = {a:.3f}*T_amb + {b_:.2f}  bar")
print(f"  max deviation from the true optimum over 25-42 C: "
      f"{np.max(np.abs(P - (a*T + b_))):.2f} bar")
# Below the critical temperature the high side can condense, so there is no
# transcritical optimum and the search returns the lower bound. Fitting through
# those points flatters nothing - refit on the transcritical range only.
mask = T >= 32
a2_, b2_ = np.polyfit(T[mask], P[mask], 1)
print(f"\n  BUT below 31 C the cycle is SUBCRITICAL and the search just returns")
print(f"  the lower bound, so those points are not real optima. Refitting on")
print(f"  the transcritical range only (T >= 32 C):")
print(f"    P_gc,opt = {a2_:.3f}*T_amb + {b2_:.2f} bar,  max deviation "
      f"{np.max(np.abs(P[mask] - (a2_*T[mask] + b2_))):.2f} bar")
print("  Use the subcritical branch below 31 C: float the high side on the")
print("  condensing temperature, exactly as you would for any other refrigerant.")


In [ ]:
fig, (a1,a2) = plt.subplots(1,2, figsize=(11.8,4.2))
a1.plot(T, P, "o-", lw=2.4, color=am.ORANGE, label="true optimum")
a1.plot(T, a*T + b_, "--", lw=1.8, color=am.MUTED, label=f"{a:.2f}·T + {b_:.1f}")
a1.set_xlabel("ambient temperature  (°C)"); a1.set_ylabel("optimum P_gc  (bar)")
a1.set_title("The optimum locus is very nearly linear"); a1.legend(fontsize=9)
a2.plot(T, [r_["COP at optimum"] for r_ in rows], "o-", lw=2.4, color=am.NAVY)
a2.axvline(am.C(crit["T"]), color="#B03A2E", ls="--")
a2.text(am.C(crit["T"])+0.3, min(r_["COP at optimum"] for r_ in rows)+0.05,
        " critical\n temperature", color="#B03A2E", fontsize=9)
a2.set_xlabel("ambient temperature  (°C)"); a2.set_ylabel("COP at the optimum")
a2.set_title("Chennai punishes CO2 in a way Europe does not")
plt.tight_layout(); plt.show()
print("  Below 31 C the high side can still condense and CO2 does well.")
print("  Above it, the cycle is transcritical and the COP falls away. That is")
print("  why parallel compression and ejectors exist, and why 'CO2 is the")
print("  natural refrigerant' needs a climate qualifier.")


## 5. The workbook

In [ ]:
sweep_rows = [booster(float(P), T_amb=am.K(35)) for P in Ps]
path = am.to_excel("AM5061_D11_CO2Booster.xlsx",
    {"Optimum locus": rows, "P_gc sweep at 35 C": sweep_rows},
    title="AM5061 D-11 . Transcritical CO2 booster, Chennai",
    summary=[("Refrigerant", F, ""),
             ("CO2 critical point", f"{am.C(crit['T']):.2f} C / {crit['p']/1e5:.2f} bar", ""),
             ("LT evaporating", am.C(T_LT), "C"), ("MT evaporating", am.C(T_MT), "C"),
             ("LT load", Q_LT/1e3, "kW"), ("MT load", Q_MT/1e3, "kW"),
             ("Receiver pressure", p_rec/1e5, "bar"),
             ("Gas cooler approach", 3.0, "K"),
             ("Optimum P_gc at 35 C", booster(P_opt)["P_gc, bar"], "bar"),
             ("COP at that optimum", booster(P_opt)["COP"], "-"),
             ("Rule of thumb slope", a, "bar/K"),
             ("Rule of thumb intercept", b_, "bar")],
    sources=[("CO2 properties", "CoolProp 8.0.0, Span & Wagner (1996) EOS"),
             ("Compressor efficiencies", "LT 0.65, MT 0.70 isentropic - ASSUMED"),
             ("Cabinet loads", "30 kW frozen, 90 kW chilled - AM5061 brief D-11"),
             ("Architecture", "booster with flash-gas bypass to MT suction")])
print("written:", path)


## What to hand in

1. COP against P_gc at design ambient, with the optimum marked.
2. The **optimum locus** over 25–42 °C, and your fitted control rule.
3. The discharge temperature at the optimum — check it against the compressor's
   limit, typically about 140 °C. An optimum you cannot run is not an optimum.
4. One paragraph on what happens to this system as ambient crosses **31 °C**,
   and what hardware you would add to fix it.
5. The workbook.

**A note on the control rule.** A linear fit is what actually goes into a
supermarket controller. Report the maximum COP you give up by using it instead
of solving the optimisation live — if that penalty is small, the simple rule
wins, and saying so is the engineering judgement being assessed.
